In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [2]:
fake = pd.read_csv("../dataset/Fake.csv")
true = pd.read_csv("../dataset/True.csv")

print("Fake shape:", fake.shape)
print("True shape:", true.shape)

Fake shape: (23481, 4)
True shape: (21417, 4)


In [3]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [4]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [5]:
fake["label"] = 0
true["label"] = 1

In [6]:
df = pd.concat([fake, true], axis=0)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.shape

(44898, 5)

In [7]:
fake_subset = df[df["label"] == 0].sample(1000, random_state=42)
true_subset = df[df["label"] == 1].sample(1000, random_state=42)

df_balanced = pd.concat([fake_subset, true_subset])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

df_balanced.shape

(2000, 5)

In [8]:
df_balanced["content"] = df_balanced["title"] + " " + df_balanced["text"]

In [9]:
df_balanced = df_balanced[["content", "label"]]
df_balanced.head()

,content,label
0,Trump says U.S. upholds and sticks to 'one Chi...,1
1,LOL! Marshawn Lynch’s Mommy Comes To Her Son’s...,0
2,Casualties in explosion at airfield near Kabul...,1
3,PROFESSOR Who Called Election Of President Tru...,0
4,"Pennsylvania state senator, Democratic officia...",1


In [10]:
import nltk
import spacy
from nltk.tokenize import sent_tokenize, word_tokenize

nlp = spacy.load("en_core_web_sm")

In [23]:
sample_text = df_balanced["content"].iloc[0]

print("Original Text:\n")
print(sample_text[:500])

print("\nSentences:\n")
print(sent_tokenize(sample_text)[:3])

print("\nWords:\n")
print(word_tokenize(sample_text)[:20])

Original Text:

Trump says U.S. upholds and sticks to 'one China' policy: Xinhua BEIJING (Reuters) - The United States government upholds and sticks to the  one China  policy, U.S. President Donald Trump told Chinese President Xi Jinping on Thursday during talks in Beijing, China s official Xinhua news agency reported. As president-elect, Trump broke with protocol and accepted a congratulatory phone call from the Taiwanese President Tsai Ing-wen in December, angering China, which claims the self-ruled island as

Sentences:

["Trump says U.S. upholds and sticks to 'one China' policy: Xinhua BEIJING (Reuters) - The United States government upholds and sticks to the  one China  policy, U.S. President Donald Trump told Chinese President Xi Jinping on Thursday during talks in Beijing, China s official Xinhua news agency reported.", 'As president-elect, Trump broke with protocol and accepted a congratulatory phone call from the Taiwanese President Tsai Ing-wen in December, angering China, wh

In [12]:
def lemmatize_text(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc])

In [13]:
print("\nLemmatized:\n")
print(lemmatize_text(sample_text)[:500])


Lemmatized:

Trump say U.S. uphold and stick to ' one China ' policy : Xinhua BEIJING ( Reuters ) - the United States government uphold and stick to the   one China   policy , U.S. President Donald Trump tell chinese President Xi Jinping on Thursday during talk in Beijing , China s official Xinhua news agency report . as president - elect , Trump break with protocol and accept a congratulatory phone call from the taiwanese President Tsai Ing - wen in December , anger China , which claim the self - rule islan


In [14]:
df_balanced["clean_text"] = df_balanced["content"].apply(lemmatize_text)

In [15]:
df_balanced.head()

,content,label,clean_text
0,Trump says U.S. upholds and sticks to 'one Chi...,1,Trump say U.S. uphold and stick to ' one China...
1,LOL! Marshawn Lynch’s Mommy Comes To Her Son’s...,0,LOL ! Marshawn Lynch ’s Mommy come to her Son ...
2,Casualties in explosion at airfield near Kabul...,1,casualty in explosion at airfield near Kabul :...
3,PROFESSOR Who Called Election Of President Tru...,0,PROFESSOR who call Election of President Trump...
4,"Pennsylvania state senator, Democratic officia...",1,"Pennsylvania state senator , democratic offici..."


In [16]:
def extract_linguistic_features(text):
    doc = nlp(text)
    
    total_tokens = len(doc)
    
    superlatives = 0
    proper_nouns = 0
    first_person_pronouns = 0
    third_person_pronouns = 0
    exclamation_count = 0
    
    for token in doc:
        # Superlatives
        if token.tag_ in ["JJS", "RBS"]:
            superlatives += 1
        
        # Proper Nouns
        if token.tag_ == "NNP":
            proper_nouns += 1
        
        # Pronouns
        if token.text.lower() in ["i", "we", "me", "us"]:
            first_person_pronouns += 1
            
        if token.text.lower() in ["he", "she", "they", "him", "her", "them"]:
            third_person_pronouns += 1
        
        # Exclamation marks
        if token.text == "!":
            exclamation_count += 1
    
    return {
        "superlative_ratio": superlatives / total_tokens if total_tokens > 0 else 0,
        "proper_noun_ratio": proper_nouns / total_tokens if total_tokens > 0 else 0,
        "first_person_ratio": first_person_pronouns / total_tokens if total_tokens > 0 else 0,
        "third_person_ratio": third_person_pronouns / total_tokens if total_tokens > 0 else 0,
        "exclamation_ratio": exclamation_count / total_tokens if total_tokens > 0 else 0
    }

In [17]:
sample_features = extract_linguistic_features(df_balanced["content"].iloc[0])
sample_features

{'superlative_ratio': 0.0,
 'proper_noun_ratio': 0.27,
 'first_person_ratio': 0.0,
 'third_person_ratio': 0.0,
 'exclamation_ratio': 0.0}

In [18]:
features_df = df_balanced["content"].apply(extract_linguistic_features)
features_df = pd.DataFrame(features_df.tolist())

df_balanced = pd.concat([df_balanced, features_df], axis=1)

df_balanced.head()

,content,label,clean_text,superlative_ratio,proper_noun_ratio,first_person_ratio,third_person_ratio,exclamation_ratio
0,Trump says U.S. upholds and sticks to 'one Chi...,1,Trump say U.S. uphold and stick to ' one China...,0.000000,0.270000,0.000000,0.000000,0.000000
1,LOL! Marshawn Lynch’s Mommy Comes To Her Son’s...,0,LOL ! Marshawn Lynch ’s Mommy come to her Son ...,0.000000,0.209346,0.001869,0.033645,0.016822
2,Casualties in explosion at airfield near Kabul...,1,casualty in explosion at airfield near Kabul :...,0.000000,0.104167,0.000000,0.000000,0.000000
3,PROFESSOR Who Called Election Of President Tru...,0,PROFESSOR who call Election of President Trump...,0.004598,0.156322,0.006897,0.016092,0.000000
4,"Pennsylvania state senator, Democratic officia...",1,"Pennsylvania state senator , democratic offici...",0.000000,0.179487,0.000000,0.000000,0.000000


In [20]:
df_balanced.groupby("label")[[
    "superlative_ratio",
    "proper_noun_ratio",
    "first_person_ratio",
    "exclamation_ratio"
]].mean()

,superlative_ratio,proper_noun_ratio,first_person_ratio,exclamation_ratio
label,,,,
0,0.001901,0.138542,0.008185,0.004424
1,0.001817,0.140457,0.004351,0.000176


In [22]:
from nltk.tokenize import sent_tokenize, word_tokenize

def average_sentence_length(text):
    sentences = sent_tokenize(text)
    if len(sentences) == 0:
        return 0
    total_words = sum(len(word_tokenize(sent)) for sent in sentences)
    return total_words / len(sentences)

df_balanced["avg_sentence_length"] = df_balanced["content"].apply(average_sentence_length)

In [24]:
df_balanced.groupby("label")["avg_sentence_length"].mean()

label
0    33.897775
1    32.577314
Name: avg_sentence_length, dtype: float64

In [25]:
def all_caps_ratio(text):
    words = text.split()
    if len(words) == 0:
        return 0
    caps = sum(1 for w in words if w.isupper() and len(w) > 1)
    return caps / len(words)

df_balanced["all_caps_ratio"] = df_balanced["content"].apply(all_caps_ratio)

In [26]:
df_balanced.groupby("label")["all_caps_ratio"].mean()

label
0    0.045857
1    0.019076
Name: all_caps_ratio, dtype: float64

In [27]:
def question_ratio(text):
    return text.count("?") / len(text) if len(text) > 0 else 0

df_balanced["question_ratio"] = df_balanced["content"].apply(question_ratio)

In [28]:
df_balanced.groupby("label")["question_ratio"].mean()

label
0    0.000614
1    0.000034
Name: question_ratio, dtype: float64

In [29]:
df_balanced.to_csv("../dataset/processed_fake_news.csv", index=False)